# Notebook 03 — limited development IBI variability pilot

**IBI variability—normal-beat status unverified.** This checkpoint calculates five authorized measures on ten deterministically selected runs per interval hypothesis. Runs require both timestamp span and supported union ≥240 seconds inside a Screen-eligible five-minute window; these are not complete five-minute HRV recordings.

This notebook displays only immutable aggregate artifacts and an explicitly synthetic arithmetic example. The guarded CLI performed the bounded real calculation after publishing selection. No participant rows, IDs, outcomes or absolute recording dates are displayed. All 32 final-test participants remain sealed.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from garmin_preprocessing.common import verify_bundle  # noqa: E402 — local source path
from garmin_hrv_input.synthetic import independent_reference  # noqa: E402 — local source path

PUBLIC = ROOT / "reports/ibi_pilot/78af22e7f0ead75b"
verify_bundle(PUBLIC)

def aggregate(name):
    return json.loads((PUBLIC / f"{name}.json").read_text())

print("Verified immutable aggregate report:", PUBLIC.relative_to(ROOT))

## Selection and corrected observation windows

The protocol was fixed before feature calculation: deduplicate identical source bounds and group overlapping source-row views without merging their intervals. Choose the earliest existing qualifying run per component, earliest per participant, then earliest ten per hypothesis. No outcome, calculated value or package acceptance enters selection.

An invariant caught signed 16-bit overflow in the old long-lookback window grid. Offsets overflowed from step 547. The pilot derives cutoff−60×int64(step), re-evaluates the unchanged Screen policy at affected windows, and preserves old bundles. Historical 12/24-hour counts are superseded by this census. Immediate through six-hour windows are unaffected.

Both hypotheses retain actual timestamps and original adjacency. Whole hypothetical intervals must lie inside [window start, window end); intervals crossing a boundary stay recorded as excluded and are never clipped or moved. An interval endpoint can meet the right boundary, but its timestamp must be strictly before it. Rejected records and gaps sever runs; there is no cleaning, interpolation or manufactured adjacency.

In [ ]:
coverage = pd.DataFrame(aggregate("feasibility"))
display(coverage[["scenario", "lookback_minutes", "ema_denominator", "participant_denominator",
                  "qualifying_ema", "qualifying_participants", "qualifying_window_instances",
                  "overlapping_source_episodes", "unique_supported_minutes"]])
display(Image(filename=str(PUBLIC / "feasibility.png")))

## Immediate losses and earlier availability

2,196 of 3,706 immediate windows have no recorded IBI; 1,510 have recordings. Only four end/five start windows pass Screen, which allows a two-minute uninterrupted run. The pilot's two 240-second requirements leave one per hypothesis. Short runs, gaps and insufficient support dominate the overlapping rejection counts; disabling the provisional jump check alone did not improve Screen retention.

Earlier qualifying runs reach 334 EMAs/51 participants for interval-end and 262 EMAs/40 participants for interval-start within 24 hours. These are full feasibility counts, not the 20-run calculation sample. Overlapping views are not independent observations; unique supported minutes use interval unions, not five times the number of windows. One qualifying segment is not complete 24-hour coverage.

In [ ]:
display(Image(filename=str(PUBLIC / "immediate_attrition.png")))
display(pd.DataFrame(aggregate("duration_summary")))
display(pd.DataFrame([aggregate("sample_overlap")]))

## What the five measures mean — synthetic arithmetic

Mean and median summarize interval length. Sample SDNN summarizes dispersion using N−1; the name does not establish normal-beat status. RMSSD measures the size of changes between original valid adjacent intervals. pNN50 is 100×number of absolute changes strictly greater than 50 ms divided by valid-pair count. P=N−1 within one intact run; no pair crosses a rejected record.

The following five intervals are **synthetic**, not participant data and not duration-eligible pilot input. Values are milliseconds; pNN50 is percent.

In [ ]:
synthetic_intervals_ms = [800, 900, 1000, 950, 850]
synthetic = independent_reference(synthetic_intervals_ms)
features = ["mean_ibi_ms", "median_ibi_ms", "sdnn_sample_ms", "rmssd_ms", "pnn50_pairs_percent"]
display(pd.DataFrame([{"label": "SYNTHETIC arithmetic example", **{f: synthetic[f] for f in features}}]))
assert synthetic["mean_ibi_ms"] == synthetic["median_ibi_ms"] == 900
assert synthetic["pnn50_pairs_percent"] == 75
print("Differences: [100, 100, -50, -100] ms; exactly 50 ms does not count.")

## Real pilot aggregates and package comparison

These tables summarize ten real selected runs per hypothesis and display no participant-level rows. Original run durations and ages are preserved in the ignored dataset; the table above gives aggregate min/median/max.

HeartPy 1.2.7 uses its approved lower-level interval/difference interface. Sample-SD conversion and fraction-to-percent conversion match the independent reference; no HeartPy median is available. NeuroKit2 0.2.13 requires end anchoring and compatible actual spacing; blocked inputs are not repaired. The original synthetic-only adapters remain unchanged. Package internals produce some ancillary time-domain quantities, but only the five authorized outputs are retained. No spectral/nonlinear routines run.

In [ ]:
display(pd.DataFrame(aggregate("feature_summary")))
display(pd.DataFrame(aggregate("package_summary")))
print("Numerical agreement does not verify normal beats, acquisition semantics or physiological validity.")

## Recommendation and reproducibility

Obtain provider clarification before expanding physiological HRV use: acquisition versus receipt timestamps, interval start/end anchoring, batching/resolution, sensor modality, confirmed units, missed-beat representation and normal-beat annotations. Earlier runs support bounded numerical work but are selective and often hours away from the EMA. A separately specified HR/activity effort can proceed while IBI questions remain unresolved.

Read `reports/ibi_pilot/78af22e7f0ead75b/report.md` and `reports/ibi_pilot_protocol.md` for exact conditions, aggregate findings and limitations. Selection lives in ignored `private/ibi_pilot/selection/52eb55d251e1d5e1/`; the 20-row output is `private/ibi_pilot/calculation/59b11baba770afb0/run_level.parquet`.

To reproduce the frozen selection and bounded calculation, using the current environment and configured `GARMIN_REFERENCE_PROJECT`:

```sh
PYTHONPATH=src uv run python -m garmin_ibi_pilot.pilot select
PYTHONPATH=src uv run python -m garmin_ibi_pilot.pilot calculate \
  --selection "$PWD/private/ibi_pilot/selection/52eb55d251e1d5e1" \
  --execute-authorized-pilot
```

Existing completed versions are hash-verified and reused; changed inputs create a new version rather than overwrite. The calculation flag authorizes this fixed protocol only. No PSS, EMA feature aggregation, models, imputation, commit or push is performed. Source notebook outputs stay empty; execution belongs under ignored `outputs/`.